# Sequence-to-sequence (encoder–decoder) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Encode a variable-length source into state, then decode a variable-length target
The cells show encoder compression, decoder recurrence, teacher forcing, exposure bias, and an end-to-end reverse-sequence toy task.

In [ ]:
src=np.array([2.,1.,3.]); h=0; hs=[]
for x in src: h=np.tanh(0.6*h+0.4*x); hs.append(h)
plt.figure(figsize=(6,3)); plt.plot(hs,'-o'); plt.title('Encoder compresses source into final state')
assert hs[-1]>hs[0]

In [ ]:
h=0.9; outs=[]
for _ in range(4): h=np.tanh(0.7*h); outs.append(h)
plt.figure(figsize=(6,3)); plt.plot(outs,'-o'); plt.title('Decoder unfolds target tokens from state')
assert len(outs)==4 and outs[-1]<outs[0]

In [ ]:
target=np.array([3,1,2]); pred=np.array([3,2,2]); teacher=(pred==target).astype(int)
plt.figure(figsize=(5,3)); plt.bar(['t1','t2','t3'],teacher); plt.title('Teacher forcing compares each predicted next token')
assert teacher.sum()==2

In [ ]:
err=0.1; steps=np.arange(1,11); survive=(1-err)**steps
plt.figure(figsize=(6,3)); plt.plot(steps,survive,'-o'); plt.title('Exposure bias compounds autoregressive errors')
assert survive[-1]<survive[0]

In [ ]:
src=np.array([1,2,3]); tgt=src[::-1]
plt.figure(figsize=(5,3)); plt.plot(src,'-o',label='source'); plt.plot(tgt,'-s',label='target'); plt.legend(); plt.title('Toy seq2seq: reverse the sequence')
assert tgt.tolist()==[3,2,1]